CSV
 ↓
Pandas DataFrame
 ↓
X, y
 ↓
Train/Test Split
 ↓
Scaling
 ↓
NumPy → Tensor
 ↓
Neural Network
 ↓
Forward Pass
 ↓
Loss
 ↓
Backward Pass
 ↓
Optimizer
 ↓
Epochs
 ↓
Evaluation
 ↓
Prediction

#### **Dataset Load**

In [24]:
import numpy as np
import pandas as pd

df = pd.read_csv(r"E:\Pradhumn- DS\Deep Learning with PyTorch\data\data.csv")
df = pd.DataFrame(df)
print(df.shape)
print(df.head())

(569, 33)
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  

In [25]:
# removing unnecessary columns
df = df.drop(columns=["id", "Unnamed: 32"])

#### **Feature and Target Creation**

In [26]:
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

In [27]:
y = y.map({
    "B": 0,
    "M": 1
})

#### **Train Test Split**

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

#### **Feature scaling**

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#### **Numpy To Tensor**

In [30]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)

In [31]:
print(X_train.shape)
print(y_train.shape)

torch.Size([455, 30])
torch.Size([455])


#### **Making Neural Network**

In [ ]:
import torch.nn as nn

model = nn.Sequential(

    # Input Layer (30 input features) → First Hidden Layer (64 neurons)
    nn.Linear(30, 64), #nn.Linear(in_features, out_features)

    # Activation Function for First Hidden Layer
    nn.ReLU(),

    # First Hidden Layer (64 neurons) → Second Hidden Layer (32 neurons)
    nn.Linear(64, 32),

    # Activation Function for Second Hidden Layer
    nn.ReLU(),

    # Second Hidden Layer (32 neurons) → Output Layer (1 neuron)
    nn.Linear(32, 1)
)

In [33]:
loss_function = nn.BCEWithLogitsLoss()

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(), #optimizer will update the parameters of the model
    lr=0.001
)

# Adam ek class hai.
# torch.optim.Adam(...) → Adam class ka object create karta hai.
# Hum us object ko "optimizer" naam dete hain.
#
# model.parameters() → model ke trainable weights aur biases
# optimizer ko provide karta hai.
#
# Isliye optimizer ko pata hota hai ki kin parameters
# (weights/biases) ko update karna hai.
#
# optimizer = ...
# ↑
# "optimizer" ek OBJECT hai
# jo Adam CLASS se bana hai.

In [ ]:
epochs = 500

for epoch in range(epochs):

    # Forward Pass
    y_pred = model(X_train) #giving the input to the model and getting the output. y_pred is the output of the model. it gives us raw scores (logits) for each sample in the training set.

    # Calculate Loss
    loss = loss_function(
        y_pred.squeeze(),
        y_train
    )
    # y_pred = model ki prediction (ŷ)
    # y_train = actual target (y)
    #
    # Loss function prediction aur actual y ko compare
    # karke batata hai ki model ki prediction kitni wrong hai.
    #
    # Training ke time actual y = y_train hota hai.
    # y_test training mein use nahi hota.
    #
    # y_pred + y_train → Loss


    # Remove old gradients
    optimizer.zero_grad()
        # PyTorch gradients ko automatically accumulate karta hai.
    # Isliye previous iteration ke gradients ko clear karte hain.
    #
    # IMPORTANT:
    # zero_grad() weights ko reset nahi karta.
    # Sirf previously stored gradients ko clear karta hai.
    #
    # Gradient = current loss ke basis par
    # weight ko kis direction mein change karna hai,
    # uski information.

    # Backpropagation. This is used to calculate the gradients of the loss with respect to the model's parameters. It computes the gradients for each parameter in the model based on the loss value.
    loss.backward()
    
    # Backpropagation ke through loss ke respect mein
    # har trainable parameter ka gradient calculate hota hai.
    #
    # Example:
    #
    # ∂Loss/∂W
    #
    # Gradient batata hai ki weight ko kis direction mein
    # change karne par loss kam hoga.
    #
    # IMPORTANT:
    # loss.backward() weights ko update NAHI karta.
    # Ye sirf gradients calculate karta hai.


    # Update weights
    optimizer.step() #Wnew ​= Wold ​− learning_rate × gradient
    # "optimizer" ek Adam class ka OBJECT hai.
    # ".step()" Adam class ka METHOD hai.
    #
    # optimizer.step()
    # → Adam optimizer ka step() method call ho raha hai.
    #
    # Ye calculated gradients ka use karke
    # model ke weights aur biases ko update karta hai.
    #
    # Conceptually:
    #
    # W_new = W_old - learning_rate × gradient
    #
    # Note:
    # Adam mein actual update formula isse more sophisticated
    # hota hai, lekin basic idea yahi hai:
    # gradient ki help se weights update hote hain.
    #
    # optimizer ke paas model.parameters() ka reference hota hai,
    # isliye step() directly model ke parameters ko update karta hai.
    #
    # IMPORTANT:
    # Updated weight ko optimizer se alag se model mein
    # bhejne ki zarurat nahi hoti.
    #
    # optimizer → model parameters ko already refer kar raha hai.
    #
    # optimizer.step()
    
    # → model ke CURRENT weights update
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}, Loss: {loss.item():.4f}"
        )
        6. REPEAT
    # --------------------------------------------------------

    # Updated weights ke saath next iteration mein
    # model dobara prediction karega.
    #
    # Complete training cycle:
    #
    # Prediction
    #     ↓
    # Loss Calculation
    #     ↓
    # Old Gradients Clear
    #     ↓
    # Gradient Calculation
    #     ↓
    # Weights Update
    #     ↓
    # Repeat
#Prediction → Loss calculate → Gradient calculate → Weights update → Repeat

Epoch 10, Loss: 0.0020
Epoch 20, Loss: 0.0019
Epoch 30, Loss: 0.0017
Epoch 40, Loss: 0.0016
Epoch 50, Loss: 0.0015
Epoch 60, Loss: 0.0014
Epoch 70, Loss: 0.0014
Epoch 80, Loss: 0.0013
Epoch 90, Loss: 0.0012
Epoch 100, Loss: 0.0012
Epoch 110, Loss: 0.0011
Epoch 120, Loss: 0.0010
Epoch 130, Loss: 0.0010
Epoch 140, Loss: 0.0010
Epoch 150, Loss: 0.0009
Epoch 160, Loss: 0.0009
Epoch 170, Loss: 0.0008
Epoch 180, Loss: 0.0008
Epoch 190, Loss: 0.0008
Epoch 200, Loss: 0.0007
Epoch 210, Loss: 0.0007
Epoch 220, Loss: 0.0007
Epoch 230, Loss: 0.0007
Epoch 240, Loss: 0.0006
Epoch 250, Loss: 0.0006
Epoch 260, Loss: 0.0006
Epoch 270, Loss: 0.0006
Epoch 280, Loss: 0.0005
Epoch 290, Loss: 0.0005
Epoch 300, Loss: 0.0005
Epoch 310, Loss: 0.0005
Epoch 320, Loss: 0.0005
Epoch 330, Loss: 0.0005
Epoch 340, Loss: 0.0005
Epoch 350, Loss: 0.0004
Epoch 360, Loss: 0.0004
Epoch 370, Loss: 0.0004
Epoch 380, Loss: 0.0004
Epoch 390, Loss: 0.0004
Epoch 400, Loss: 0.0004
Epoch 410, Loss: 0.0004
Epoch 420, Loss: 0.0004
E